# Getting started

SciLogic stores the argument of a paper as an **encoding**: a YAML file with typed statements, typed relations between them, and an assessment of every claim. This tutorial loads a small encoding, looks inside it, and runs the analysis that `claim-chain` provides.

You get an encoding from a PDF with the `claimchain` converter, which asks Claude to read the paper:

```bash
pip install git+https://github.com/lab-mids/claim-chain
export ANTHROPIC_API_KEY=...
claimchain paper.pdf -o paper.yaml
```

This takes a few minutes per paper. Here we skip it and start from an encoding that is already in the repository: a toy paper, invented for this tutorial, that is small enough to read in full.

In [ ]:
import sys
from collections import Counter

import pandas as pd
import yaml
from IPython.display import Markdown

sys.path.insert(0, "../scripts")
import render  # claim-chain's analysis and rendering module (scripts/render.py)

## Load an encoding

An encoding is plain YAML, so `yaml.safe_load` is all you need. The analysis functions in `render.py` work on the resulting dictionary.

In [ ]:
with open("data/example_encoding.yaml") as f:
    chain = yaml.safe_load(f)

pub = chain["publication"]
print(f"{pub['title']} ({', '.join(pub['authors'])}, {pub['year']})")
print(f"Statements: {len(chain['statements'])}")
print(f"Relations:  {len(chain['relations'])}")
print(f"Types:      {dict(Counter(s['statement_type'] for s in chain['statements']))}")

## Statements

Each statement has an id, a type, the part of the paper it comes from, and its text. Claims and hypotheses also carry the annotator's assessment.

In [ ]:
pd.DataFrame([
    {
        "id": render.short(s["id"]),
        "type": s["statement_type"],
        "part": s.get("paper_part", ""),
        "verdict": (s.get("assessment") or {}).get("validity", ""),
        "text": s["text"],
    }
    for s in chain["statements"]
]).style.hide(axis="index")

Here is one statement in full. The observation records the numbers it rests on (`values`) and a **derived check**: a formula that `render.py` recomputes from those values.

In [ ]:
statements = {s["id"]: s for s in chain["statements"]}
print(yaml.safe_dump(statements["toy:OBS1"], sort_keys=False, allow_unicode=True))

## Relations

A relation links a subject statement to an object statement with a predicate. It also says who asserts the link: the authors, or the annotator who reconstructed the argument.

In [ ]:
print(yaml.safe_dump(chain["relations"][4], sort_keys=False))

pd.DataFrame([
    {"subject": render.short(r["subject"]), "predicate": r["predicate"],
     "object": render.short(r["object"]), "asserted by": r["asserted_by"]}
    for r in chain["relations"]
]).style.hide(axis="index")

## The vocabulary

The statement and relation types are defined in the schema (`schema/claim_chain.yaml`). The tables below are read from it, so they always match the format you are using.

In [ ]:
with open("../schema/claim_chain.yaml") as f:
    schema = yaml.safe_load(f)

def vocabulary(enum):
    values = schema["enums"][enum]["permissible_values"]
    return pd.DataFrame(
        [{enum: k, "meaning": (v or {}).get("description", "")} for k, v in values.items()]
    ).style.hide(axis="index")

print("schema version", schema["version"])
vocabulary("StatementType")

In [ ]:
vocabulary("RelationType")

## The argument map

`render.mermaid` turns the encoding into a [Mermaid](https://mermaid.js.org) flowchart. Thick arrows are strong support, dashed arrows qualify or weaken, and the marks after the ids are the verdicts (✓ YES, partial, ? INCONCLUSIVE).

In [ ]:
Markdown(f"```{{mermaid}}\n{render.mermaid(chain)}\n```")

## Scores and checks

Three questions you can answer for any encoding:

1. **Logic score:** the mean verdict over all claims and hypotheses (YES = 1, PARTIAL = 0.5, INCONCLUSIVE = 0.25, NO = 0).
2. **Derived verdicts:** what the verdicts would be if they followed mechanically from the links (weakest-link rule). Disagreements with the annotator are worth a second look.
3. **Recomputed checks:** do the numbers recorded in the encoding give the results the paper reports?

In [ ]:
score, counts, _ = render.logic_score(chain)
derived, disagreements = render.derived_score(chain)
coverage, n_obligations = render.obligation_coverage(chain)

print(f"Logic score:          {score:.2f}  {counts}")
print(f"Derived score:        {derived:.2f}")
for sid, annotator, rule in disagreements:
    print(f"  {render.short(sid)}: annotator says {annotator}, links give {rule}")
print(f"Obligation coverage:  {coverage:.2f} of {n_obligations} proof obligations")

for sid, check, status, computed in render.run_checks(chain):
    print(f"Check in {render.short(sid)}: {check['expression']} = {computed:.1f} {check.get('unit', '')} "
          f"(paper: {check['result']}) -> {status}")

The disagreement on C1 is informative: the annotator judged the hardness increase demonstrated (YES), but the authors' own limitation LIM1 *qualifies* C1, and the weakest-link rule caps any qualified claim at PARTIAL. Whether one Al content is enough for the claim as worded is exactly the kind of question a reviewer should decide.

## Proof obligations

Every assessed claim lists what would have to be shown for it to hold, and which statements discharge each obligation. Open obligations are the paper's to-do list.

In [ ]:
for s in chain["statements"]:
    for po in (s.get("assessment") or {}).get("proof_obligations", []):
        by = ", ".join(render.short(x) for x in po.get("discharged_by", [])) or "-"
        print(f"{render.short(s['id']):4} {po['obligation_status']:10} {po['text']}  (by: {by})")

## The full report

For a complete human-readable report, run `render.py` on the command line. It writes a Markdown version of the chain, the three-part report (logic, limitations, missing evidence), the argument maps, and a one-page summary chart:

```bash
python scripts/render.py docs/data/example_encoding.yaml generated/
```

## Next steps

- [Peer review](01-peer-review.ipynb) applies this to a real paper, Zhuang, Chen and Carter (2017), and shows how verdicts, recomputed checks and proof obligations guide a review.
- [Research opportunities](02-research-opportunities.ipynb) collects the open items of ten recent preprints and turns them into research ideas.